# Initial Cleaning

***Based on the initial EDA, we confirmed that the dataset contains no duplicate entries. However, several issues remain that require attention before proceeding to the main analysis.***

***First, a number of columns are stored in incorrect data types and will be converted to their appropriate types to ensure consistency and readability throughout the analysis.***

***Second, certain columns suffer from extreme nullness, with `Achievements` reaching up to 99.97% null values, and the dataset as a whole exhibits a strong right skew in several numeric columns. Columns that offer little analytical value will be dropped.***

***Finally, as Steam is a large platform that hosts more than just games, the dataset likely includes non-game entries such as software, tools, and DLC; these will be filtered out to ensure our analysis focuses solely on games.***

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

script_path = Path.cwd()
data_path = script_path.parent/"data"/"raw"/"games.csv"

df = pd.read_csv(data_path, index_col=False)
df.head(5)

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",...,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...",...,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",...,0,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...
4,3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,...,0,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...


## Datatype fixing

In [2]:
df.dtypes

AppID                           int64
Name                           object
Release date                   object
Estimated owners               object
Peak CCU                        int64
Required age                    int64
Price                         float64
DiscountDLC count               int64
About the game                  int64
Supported languages            object
Full audio languages           object
Reviews                        object
Header image                   object
Website                        object
Support url                    object
Support email                  object
Windows                        object
Mac                              bool
Linux                            bool
Metacritic score                 bool
Metacritic url                  int64
User score                     object
Positive                        int64
Negative                        int64
Score rank                      int64
Achievements                  float64
Recommendati

***To investigate the dtype inconsistencies more thoroughly, the raw CSV file was opened directly in VS Code using the Rainbow CSV extension, which renders each column in a distinct color for easier visual inspection. This immediately confirmed the root cause: the column headers and their corresponding values were misaligned throughout the file, with every column shifted one position to the right relative to its actual data.***

In [3]:
pd.set_option('display.max_columns', None)
df.head(5)

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],[],NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,NaN,NaN,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],[],NaN,https://shared.akamai.steamstatic.com/store_it...,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,NaN,0,252,3,NaN,0,231,NaN,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",[],NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.facebook.com/8FloorGames/,https://www.facebook.com/8FloorGames,support@8floor.net,True,True,False,0,NaN,0,21,3,NaN,0,0,NaN,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],['Korean'],NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,NaN,yujingamesc@gmail.com,True,False,False,0,NaN,0,0,0,NaN,19,0,The game includes the following elements. 1. G...,0,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...
4,3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],['English'],NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.realityexpanded.com/books-games,https://www.realityexpanded.com,support@realityexpanded.com,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...


***This table makes the column misalignment immediately visible: the description of each game, which belongs to `About the game`, is instead appearing under `Supported languages`, confirming that the shift begins at the `About the game` column.***

***Examining the columns that precede it offers a clue as to why: every column before `DiscountDLC count` aligns correctly with its values, yet there is one
extra numeric column unaccounted for. This suggests that `DiscountDLC count` was originally two separate columns, `Discount` and `DLC count`, that were merged into one during data collection, introducing the off-by-one shift that propagates through the rest of the dataset.***

In [4]:
import io

with open(data_path, 'r', encoding='utf-8') as file:
    raw_text = file.read()

fixed_text = raw_text.replace('DiscountDLC count', 'Discount,DLC count', 1)

df = pd.read_csv(io.StringIO(fixed_text))

***The original CSV contains a missing comma in the header row, causing `DiscountDLC count` to represent two separate fields. A string replacement is applied before parsing, inserting the missing comma to produce `Discount` and `DLC count` as distinct columns. `io.StringIO` allows pandas to treat the corrected string as a file object, so the original CSV remains untouched.***


In [5]:
df[['Min owners', 'Max owners']] = df['Estimated owners'].str.split(' - ', expand=True)
df['Min owners'] = pd.to_numeric(df['Min owners'])
df['Max owners'] = pd.to_numeric(df['Max owners'])
df['Avg owners'] = (df['Min owners'] + df['Max owners']) / 2
df.drop(columns=['Estimated owners'], inplace=True)

df.head(5)

,AppID,Name,Release date,Peak CCU,Required age,Price,Discount,DLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies,Min owners,Max owners,Avg owners
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0,0.00,0,0,NaN,[],[],NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,NaN,NaN,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,0,0,0.0
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],[],NaN,https://shared.akamai.steamstatic.com/store_it...,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,NaN,0,252,3,NaN,0,231,NaN,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN,0,20000,10000.0
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",[],NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.facebook.com/8FloorGames/,https://www.facebook.com/8FloorGames,support@8floor.net,True,True,False,0,NaN,0,21,3,NaN,0,0,NaN,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN,0,20000,10000.0
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],['Korean'],NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,NaN,yujingamesc@gmail.com,True,False,False,0,NaN,0,0,0,NaN,19,0,The game includes the following elements. 1. G...,0,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,0,20000,10000.0
4,3631080,Maze Quest VR,"Apr 24, 2025",0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],['English'],NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.realityexpanded.com/books-games,https://www.realityexpanded.com,support@realityexpanded.com,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,0,20000,10000.0


In [6]:
df.dtypes

AppID                           int64
Name                           object
Release date                   object
Peak CCU                        int64
Required age                    int64
Price                         float64
Discount                        int64
DLC count                       int64
About the game                 object
Supported languages            object
Full audio languages           object
Reviews                        object
Header image                   object
Website                        object
Support url                    object
Support email                  object
Windows                          bool
Mac                              bool
Linux                            bool
Metacritic score                int64
Metacritic url                 object
User score                      int64
Positive                        int64
Negative                        int64
Score rank                    float64
Achievements                    int64
Recommendati

In [7]:
df['Release date'] = pd.to_datetime(df['Release date'], format='mixed', errors='coerce')
df['Release date'].head(10)

0   2023-08-01
1   2016-07-29
2   2019-05-06
3   2024-10-31
4   2025-04-24
5   2023-04-05
6   2025-04-08
7   2018-02-01
8   2021-05-07
9   2021-12-13
Name: Release date, dtype: datetime64[ns]

***After fixing the column misalignment and splitting `Estimated owners` into three separate columns, retaining the original `Estimated owners min` and `Estimated owners max` values and deriving a new `Estimated owners avg` for potential future use, the dtype table was re-examined.***

***Most columns now show the correct data types, confirming that the majority of the dtype inconsistencies were a direct consequence of the earlier misalignment.
`Release date` has also been successfully converted to `datetime`. However, several columns still appear as `float64` despite their values being better
represented as integers or objects, which is likely caused by the high proportion of null values across the dataset — pandas defaults to `float64` when a column contains missing values.***

***Resolving these remaining dtype issues will be the focus of the next step in this cleaning notebook.***

---

## Nullness

In [8]:
null_counts = df.isnull().sum()
null_pct = ((null_counts / len(df)) * 100).round(2)
null_summary = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
null_summary[null_summary["null_count"] > 0].sort_values(by="null_count", ascending=False)

,null_count,null_pct
Movies,122611,100.00
Score rank,122571,99.97
Metacritic url,118355,96.53
Reviews,110541,90.16
Notes,100153,81.68
Website,72935,59.48
Support url,68469,55.84
Tags,39265,32.02
Support email,22263,18.16
Categories,8953,7.30


***Due to the column shift, the `Movies` column became entirely null. Taking into account other affected columns such as `Score rank` and `Reviews`, while these metrics could potentially be useful in the analysis, the high proportion of missing values renders them unusable in their current state.***

***The primary dropping criterion is a null rate exceeding 60%, however, `Website` and `Support url` were also dropped despite falling slightly below this threshold, both offer little analytical value in evaluating a game, and their null rates remain considerably higher than most other retained columns.***

In [9]:
df =df.drop(columns=['Movies','Score rank','Metacritic url','Reviews','Notes','Website','Support url'])
df.head(5)

,AppID,Name,Release date,Peak CCU,Required age,Price,Discount,DLC count,About the game,Supported languages,Full audio languages,Header image,Support email,Windows,Mac,Linux,Metacritic score,User score,Positive,Negative,Achievements,Recommendations,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Min owners,Max owners,Avg owners
0,2539430,Black Dragon Mage Playtest,2023-08-01,0,0,0.00,0,0,NaN,[],[],https://shared.akamai.steamstatic.com/store_it...,NaN,True,False,False,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,0,0,0.0
1,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],[],https://shared.akamai.steamstatic.com/store_it...,support@mangagamer.com,True,False,False,0,0,252,3,0,231,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,0,20000,10000.0
2,1034400,Mystery Solitaire The Black Raven,2019-05-06,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",[],https://shared.akamai.steamstatic.com/store_it...,support@8floor.net,True,True,False,0,0,21,3,0,0,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,0,20000,10000.0
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,2024-10-31,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],['Korean'],https://shared.akamai.steamstatic.com/store_it...,yujingamesc@gmail.com,True,False,False,0,0,0,0,19,0,0,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,0,20000,10000.0
4,3631080,Maze Quest VR,2025-04-24,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],['English'],https://shared.akamai.steamstatic.com/store_it...,support@realityexpanded.com,True,False,False,0,0,0,0,0,0,0,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,0,20000,10000.0


## Removing non games

***As shown in `df.head(5)`, the second entry reveals a playtest application with zero values across all numeric columns, indicating it was used by developers to test the game prior to its public release. As such, it does not represent an actual consumer-facing game and would act as an outlier in the main EDA.***

***With this in mind, all non game entries, including playtests, software tools, and DLC will be filtered out before the main analysis is conducted.***

In [10]:
print(df['Tags'].value_counts())

Tags
Adventure,Casual,Hidden Object                                                                                                                                                                                                                                        273
Action,Indie                                                                                                                                                                                                                                                          249
Indie,Casual                                                                                                                                                                                                                                                          236
Action,Indie,Casual                                                                                                                                                                                  

In [11]:
print(df['Tags'].dtype)
print(df['Tags'].iloc[0])
print(df['Tags'].isna().sum())

object
nan
39265


***The initial approach was to filter by tags, as non game entries would likely carry tags indicating their application type. However, since the `Tags` column stores multiple tags as a single combined string, separating and filtering them reliably at this early cleaning stage would be unnecessarily complex.***

***Instead, the approach taken is to search for keywords in the `Name` column to identify and verify non game entries before removing them from the dataset.***

In [12]:
playtest_mask = df['Name'].str.contains('Playtest', case=False, na=False)
print(f"Playtest entries: {playtest_mask.sum()}")
print(df[playtest_mask]['Name'].head(10))
df = df[~playtest_mask]

Playtest entries: 8129
0              Black Dragon Mage Playtest
10             Codename: Warlock Playtest
24                    CyberVault Playtest
25      A Night In Omar's Burger Playtest
51                 Dark and Deep Playtest
97             Bill's Misfortune Playtest
102                 QUADRA CLASH Playtest
116                  Descendance Playtest
118    Red Solstice 2: Survivors Playtest
137                   Pain Party Playtest
Name: Name, dtype: object


In [13]:
junk_keywords = ['soundtrack', 'dlc', 'season pass', 'artbook', 
                 'expansion pack', 'bonus content', 'demo']
junk_mask = df['Name'].str.lower().str.contains('|'.join(junk_keywords), na=False)

print(f"Non game entries: {junk_mask.sum()}")
print(df[junk_mask]['Name'].head(20))

Non game entries: 461
444                                           75 Demons
587                                Evil Democracy: 1932
622                                    DEMON GAZE EXTRA
1203                                      Demon Hunters
1273                             Spellbook Demonslayers
1582                    Disturbing Forest: Demon's Path
1829                 Exoplanet: A Project Coreward Demo
2289                                Demonheart: Hunters
2438                                        Kill Demons
2547                                       Demonic Pack
2563                           降妖师Demon Subduing Master
2654                             Kunoichi Demon Slayers
3109                           Horror Engine: Tech Demo
3202                          Borderus: Angels & Demons
3316    Fallen Priestess: My Sister's Demonic Bloodline
3423                                   Demolition Party
3806                                 Demon Slayer Shion
4145                      

***When searching for keywords such as `demo`, partial matches become a problem, words like `Demon` also trigger the filter despite referring to a legitimate game title. To address this, the `re` library is used with word boundary markers (`\b`) to match keywords as standalone words only, preventing false positives from partial string matches.***

In [14]:
import re

junk_keywords = [
    r'\bsoundtrack\b',
    r'\bdlc\b', 
    r'\bseason pass\b',
    r'\bartbook\b',
    r'\bexpansion pack\b',
    r'\bbonus content\b',
    r'\bdemo\b'
]

junk_mask = df['Name'].str.lower().str.contains('|'.join(junk_keywords), na=False, regex=True)

print(f"Non game entries: {junk_mask.sum()}")
print(df[junk_mask]['Name'].head(20))

Non game entries: 79
1829                    Exoplanet: A Project Coreward Demo
3109                              Horror Engine: Tech Demo
4412                                      Final Crash Demo
5289                               Cappasity VR Store Demo
5820                                  Monster Battles Demo
8307                       Seeking Asylum: The Game (DEMO)
8446                                     Brisk Square Demo
10196                                    Stellarace (Demo)
10554                                         Reboant Demo
11791           Vagrus - The Riven Realms: Demo (Prologue)
15253                                 Multiplayer FPS Demo
16647                         The Forgotten Village (Demo)
17485                                            DLC Quest
17736                              OutcastKidsAndCity DEMO
18935                                         Proptop Demo
25316                                       CRAZY DOG DEMO
27220    Indiana Jones and the Grea

In [15]:
df = df[~junk_mask]
print(f"Rows removed: {junk_mask.sum()}")


Rows removed: 79


In [16]:
ghost_mask = (
    (df['Max owners'] == '0') &
    (df['Positive'] == 0) &
    (df['Negative'] == 0)
)
print(f"Ghost entries: {ghost_mask.sum()}")
print(df[ghost_mask]['Name'].head(10))

Ghost entries: 0
Series([], Name: Name, dtype: object)


***The keyword search approach proved successful. Ghost entry detection was then performed using `Max owners`, `Positive`, and `Negative` as indicators, yielding no genuine ghost entries in the dataset.***

***With the initial cleaning now complete , covering column realignment, dtype corrections, null based column removal, and non game entry filtering, the dataset is prepared for a more detailed and thorough EDA in the next notebook.***

---

### Quick Sanity Check

In [24]:
# Playtest-8129, Non game Entry-79, Full Dataset-122611
# 122611 - 8129 - 79 = 114403
print(f'Dataset size after cleaning: {len(df)} rows')

# "AppID", "Name", "Release date", "Estimated owners", "Peak CCU", 
#     "Required age", "Price", "DiscountDLC count", "About the game", 
#     "Supported languages", "Full audio languages", "Reviews", 
#     "Header image", "Website", "Support url", "Support email", 
#     "Windows", "Mac", "Linux", "Metacritic score", "Metacritic url", 
#     "User score", "Positive", "Negative", "Score rank", "Achievements", 
#     "Recommendations", "Notes", "Average playtime forever", 
#     "Average playtime two weeks", "Median playtime forever", 
#     "Median playtime two weeks", "Developers", "Publishers", 
#     "Categories", "Genres", "Tags", "Screenshots", "Movies"

# + "Min owners", "Max owners", "Avg owners"
# - "Movies", "Score rank", "Metacritic url", "Reviews", "Notes", "Website", "Support url"

expected_cols = ['AppID', 'Name', 'Release date', 'Peak CCU', 'Required age', 
                'Price', 'Discount', 'DLC count', 'About the game', 
                'Supported languages', 'Full audio languages', 'Header image', 
                'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 
                'User score', 'Positive', 'Negative', 'Achievements', 
                'Recommendations', 'Average playtime forever', 'Average playtime two weeks', 
                'Median playtime forever', 'Median playtime two weeks', 'Developers', 
                'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 
                'Min owners', 'Max owners', 'Avg owners']
print(set(df.columns) == set(expected_cols))



Dataset size after cleaning: 114403 rows
True


***Sanity check passed. All expected columns are present and accounted for. Everything looks good to go for the main EDA.***

---

In [ ]:
data_path = script_path.parent/"data"/"processed"/"games_clean_r1.csv"
df.to_csv(data_path, index=False)

Saved: 114403 rows
